# Lesson 5.4 — Unit 5 Recap: Monitoring with the Twin

Comparison + two divergence signals + diagnosis = a live monitor.

In [1]:
import numpy as np
from modules.module09.integration import (GreenhouseWorld, model_perception, understand,
                                          run_pipeline)
from modules.module10.twin import (DigitalTwin, GroundTruth, outcome_gap,
                                   monitor, predict, compare_futures)
def victim_and_xy(seed=1):
    w = GreenhouseWorld.demo_row(n=6, seed=seed)
    dets = model_perception(w, rng=np.random.default_rng(7))
    v = understand(dets, w)[1]['id']
    vxy = next(d['xy'] for d in dets if d['id']==v)
    return v, vxy
checks = []
v, vxy = victim_and_xy()
# monitor in sync
g = GroundTruth(GreenhouseWorld.demo_row(n=6, seed=1)); t = DigitalTwin(g.world); t.sync(g.report())
checks.append(monitor(t, g.report())['alert'] is False)
# state divergence on move
g.world.q = g.world.q + np.array([0.1,0.0])
checks.append(monitor(t, g.report())['alert'] is True)
# outcome divergence localised + directional
g2 = GroundTruth(GreenhouseWorld.demo_row(n=6, seed=1), unmodeled={v: dict(obstacle=(vxy,0.25))})
t2 = DigitalTwin(g2.world)
gap = outcome_gap(t2.simulate(), g2.run())
checks.append(gap['harvested_only_in_sim'] == [v])
print('in-sync-quiet=%s state-alert=%s outcome-localised=%s' % tuple(checks))
assert all(checks), f'FAILED: {checks}'
print(f'{sum(checks)}/{len(checks)} checks passed.')
print('All checks passed.')


in-sync-quiet=True state-alert=True outcome-localised=True
3/3 checks passed.
All checks passed.
